# 03 Consensus-Sequence Generation and Clade Annotation

**Methods mapping:** consensus-sequence generation and clade annotation.

This notebook checks the study consensus G FASTA files in `data/input/consensus/` and joins retained samples to the curated clade annotation in metadata.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'config' / 'analysis_config.yaml').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT / 'notebooks'))
import analysis_utils as au
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
rel = lambda path: Path(path).relative_to(ROOT).as_posix()
ROOT


PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes')

In [2]:
DATA_DIR, FIG_DIR = au.step_dirs('03_consensus_sequence_generation_and_clade_annotation', ROOT)
INPUTS = {
    "metadata": ROOT / "data/metadata/meta_v6_with_season_clade.csv",
    "AU_consensus_fasta": ROOT / "data/input/consensus/PRJNA1037681_all_consensus_extracted_4652-5617.fasta",
    "US_consensus_fasta": ROOT / "data/input/consensus/PRJNA1130896_all_consensus_extracted_4652-5617.fasta",
    "AU_haplotype_fasta": ROOT / "data/input/haplotypes/PRJNA1037681_extracted_4652-5617.fasta",
    "US_haplotype_fasta": ROOT / "data/input/haplotypes/PRJNA1130896_extracted_4652-5617.fasta",
}
OUTPUTS = {
    "clade_counts": DATA_DIR / "retained_sample_clade_counts.csv",
    "fasta_manifest": DATA_DIR / "consensus_and_g_sequence_file_manifest.csv",
}


def show_paths(title, paths):
    rows = []
    for name, path in paths.items():
        path = Path(path)
        rows.append({"name": name, "relative_path": rel(path), "exists": path.exists()})
    display(Markdown(f"### {title}"))
    display(pd.DataFrame(rows))

for path in OUTPUTS.values():
    Path(path).parent.mkdir(parents=True, exist_ok=True)

show_paths("Input paths", INPUTS)
show_paths("Output paths", OUTPUTS)
DATA_DIR, FIG_DIR


### Input paths

,name,relative_path,exists
0,metadata,data/metadata/meta_v6_with_season_clade.csv,True
1,AU_consensus_fasta,data/input/consensus/PRJNA1037681_a...,True
2,US_consensus_fasta,data/input/consensus/PRJNA1130896_a...,True
3,AU_haplotype_fasta,data/input/haplotypes/PRJNA1037681_...,True
4,US_haplotype_fasta,data/input/haplotypes/PRJNA1130896_...,True


### Output paths

,name,relative_path,exists
0,clade_counts,data/processed_data/03_consensus_sequenc...,True
1,fasta_manifest,data/processed_data/03_consensus_sequenc...,True


(PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes/data/processed_data/03_consensus_sequence_generation_and_clade_annotation'),
 PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes/results/figures/03_consensus_sequence_generation_and_clade_annotation'))

## Check Consensus FASTA Inputs

This cell reads the study consensus FASTA and haplotype FASTA directly and summarizes record counts and G-segment lengths.


In [3]:
rows = []
for project, cfg in au.PROJECTS.items():
    for role, path in [
        ('study_consensus_G_FASTA', au.input_consensus_fasta_path(ROOT, project)),
        ('haplotype_G_FASTA', au.input_haplotype_fasta_path(ROOT, project)),
    ]:
        recs = au.read_fasta_records(path)
        rows.append({
            'project': project,
            'dataset': cfg['label'],
            'role': role,
            'records': len(recs),
            'min_length_nt': recs['length_nt'].min(),
            'median_length_nt': recs['length_nt'].median(),
            'max_length_nt': recs['length_nt'].max(),
            'path': str(path.relative_to(ROOT)),
        })
fasta_check = pd.DataFrame(rows)
display(fasta_check)


,project,dataset,role,records,min_length_nt,median_length_nt,max_length_nt,path
0,PRJNA1037681,Australia,study_consensus_G_FASTA,38,966,966.0,966,data/input/consensus/PRJNA1037681_a...
1,PRJNA1037681,Australia,haplotype_G_FASTA,75,966,966.0,966,data/input/haplotypes/PRJNA1037681_...
2,PRJNA1130896,United States,study_consensus_G_FASTA,76,966,966.0,966,data/input/consensus/PRJNA1130896_a...
3,PRJNA1130896,United States,haplotype_G_FASTA,145,966,966.0,966,data/input/haplotypes/PRJNA1130896_...


## Rebuild Clade Counts

Clade labels come from curated metadata after the retained sample set is rebuilt from the haplotype FASTA.


In [4]:
clade_counts, fasta_manifest = au.consensus_clade_annotation(ROOT, DATA_DIR)

display(clade_counts)
display(fasta_manifest)


,project,project_label,clade,n_samples
1,PRJNA1037681,Australia,A.D.3.1,31
0,PRJNA1037681,Australia,A.D.3,5
2,PRJNA1037681,Australia,A.D.3.3,1
4,PRJNA1130896,United States,A.D.1.5,37
14,PRJNA1130896,United States,A.D.5.2,14
3,PRJNA1130896,United States,A.D.1,4
7,PRJNA1130896,United States,A.D.1.9,3
5,PRJNA1130896,United States,A.D.1.6,2
6,PRJNA1130896,United States,A.D.1.8,2
8,PRJNA1130896,United States,A.D.3,2


,project,project_label,role,path,exists,records,min_length_nt,max_length_nt,bytes
0,PRJNA1037681,Australia,study_consensus_G_FASTA,data/input/consensus/PRJNA1037681_a...,True,38,966,966,37851
1,PRJNA1037681,Australia,haplotype_G_FASTA,data/input/haplotypes/PRJNA1037681_...,True,75,966,966,77273
2,PRJNA1130896,United States,study_consensus_G_FASTA,data/input/consensus/PRJNA1130896_a...,True,76,966,966,75699
3,PRJNA1130896,United States,haplotype_G_FASTA,data/input/haplotypes/PRJNA1130896_...,True,145,966,966,149413
